# Import libraries

In [42]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load Ground Truth

In [43]:
def load_gt(gt_fname):
    with open(gt_fname) as f:
        gt = json.load(f)

    rows_gt = []
    for video_id, info in gt["database"].items():
        duration = info["duration"]
        subset = info["subset"]
        for s in info["annotations"]:
            rows_gt.append(
                {
                    "video_id": video_id,
                    "duration": duration,
                    "subset": subset,
                    "start": s["segment"][0],
                    "end": s["segment"][1],
                    "label": s["label"],
                }
            )

    return pd.DataFrame(rows_gt)

### Load full ground truth

In [44]:
gt_df = load_gt("../data/charades/annotations/charades.json")

# Load ground truth function

### Load sample ground truth (10 least frequent classes)

In [45]:
# least_sample_gt_df = load_gt("../data/charades/annotations/least_sample_charades.json")
# print(least_sample_gt_df.shape)
# least_sample_gt_df.head()

### Load sample ground truth (10 most frequent classes)

In [46]:
# most_sample_gt_df = load_gt("../data/charades/annotations/most_sample_charades.json")
# print(most_sample_gt_df.shape)
# most_sample_gt_df.head()

### Load predictions

In [47]:
with open("../exps/charades/actionformer_i3d_rgb/result_detection.json", "r") as f:
  predictions = json.load(f)

rows_predictions = []
for video_id, segments in predictions['results'].items():
  for s in segments:
    rows_predictions.append({
      'video_id': video_id,
      'start': s['segment'][0],
      'end': s['segment'][1],
      'label': s['label'],
      'score': s['score']
    })
  
predictions_df = pd.DataFrame(rows_predictions)
predictions_df.sample(50)

,video_id,start,end,label,score
5814,1XBU2,9.92,11.97,Sitting in a chair,0.0064
83826,XHYA2,5.88,7.91,Someone is eating something,0.0216
113,00T1E,-0.00,6.83,Someone is eating something,0.0109
1510,0E6H9,0.09,3.26,Someone is smiling,0.0225
13195,61IVZ,0.75,22.43,Walking through a doorway,0.4131
24998,B1AMA,3.63,5.69,Someone is standing up from somewhere,0.0190
443,02DPI,7.94,10.05,Someone is eating something,0.0193
6689,2OHTZ,15.73,17.70,Someone is smiling,0.0052
71578,TJIK7,40.73,41.95,Sitting in a chair,0.0193
78057,VW4UD,11.30,13.30,Drinking from a cup/glass/bottle,0.0021


In [48]:
predictions_df.shape

(90584, 5)

# Plot Predictions against Ground Truth

In [49]:
def plot_predictions_vs_gt(predictions_df, gt_df, video_id, confidence_threshold=0.04):
    # predictions for the video
    predicted_video = predictions_df[
        (predictions_df["video_id"] == video_id)
        & (predictions_df["score"] > confidence_threshold)
    ]
    # ground truth for the video
    video = gt_df[gt_df["video_id"] == video_id]

    # Plot predictions against ground truth
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    for _, row in predicted_video.iterrows():
        ax[0].plot([row["start"], row["end"]], [row["label"]] * 2)
    ax[0].set_title("Prediction")

    for _, row in video.iterrows():
        ax[1].plot([row["start"], row["end"]], [row["label"]] * 2)
    ax[1].set_title("Ground Truth")

    plt.xlabel("Second")
    plt.tight_layout()
    plt.show()

In [50]:
# plot_predictions_vs_gt(predictions_df, most_sample_gt_df, "0TDOP", confidence_threshold=0.3)

# Overlapping Analysis

In [51]:
def calculate_overlap_density(df, video_id):
    # Filter rows for the given video
    video_df = df[df["video_id"] == video_id].copy()

    if len(video_df) < 2:
        return 0.0  # No overlap possible with fewer than 2 segments

    # Get total video duration (assume all rows have the same duration)
    total_duration = video_df["duration"].iloc[0]

    # Build a list of (start, end) tuples
    intervals = list(zip(video_df["start"], video_df["end"]))

    # Create a timeline of second-level granularity
    timeline = [0.0] * int(total_duration * 10)  # use deci-seconds for precision

    # Mark timeline per action
    for start, end in intervals:
        for t in range(int(start * 10), int(end * 10)):
            if 0 <= t < len(timeline):
                timeline[t] += 1

    # Count overlapping time points (where >1 action occurs)
    overlap_count = sum(1 for t in timeline if t > 1)

    # Convert back to seconds
    overlap_duration = overlap_count / 10.0

    # Compute and return overlap density
    return overlap_duration / total_duration

In [52]:
# gt_df['overlap_density'] = gt_df['video_id'].apply(lambda x: calculate_overlap_density(gt_df, x))
# gt_df['overlap_density'].describe()

# Check the number of frames

In [55]:
ego_df = pd.read_csv("../data/charades/annotations/wise_annotations/CharadesEgo_v1_train_only1st.csv")
ego_df.head()

,id,subject,scene,quality,relevance,verified,script,objects,descriptions,actions,length,egocentric,charades_video
0,D3TR8EGO,2Q9D,Closet / Walk-in closet / Spear closet,7.0,7.0,Yes,A person walks into the closet and turns on th...,cup/glass/bottle;door;food;glass of water;ligh...,The person in the video walks into a room and ...,c156 3.90 12.00;c061 8.20 12.50;c106 9.90 18.4...,31.79,Yes,1K0SU
1,65JW9EGO,ZG1V,Living room,3.0,7.0,Yes,"A person was walking towards the sofa, and tak...",couch;floor;food;glass;pillow;table;window,A person walks into a room and grabs a pillow ...,c109 19.90 24.29;c107 16.80 23.80;c106 10.10 2...,27.38,Yes,NaN
2,3UML4EGO,0AC0,Home Office / Study (A room in a house used fo...,7.0,7.0,Yes,A person is sitting in a chair in their home o...,chair;coffee;cup;desk;phone;table,person is sitting on a chair playing on the ph...,c107 16.50 22.04;c110 15.30 21.00;c109 23.10 2...,26.75,Yes,NaN
3,6U5TEEGO,5LWB,Bathroom,6.0,6.0,Yes,One person washes a mirror and tidies up while...,blanket;cloth;mirror;sofa/couch;towel,A person cleans mirror and another person take...,c073 14.90 20.00;c095 0.00 15.90;c070 10.20 20...,21.75,Yes,YFZRG
4,KCSBQEGO,UN1T,Living room,6.0,5.0,Yes,A smiling person is fixing a window in their l...,tool;towel;window,The person appears to be fixing a window in th...,NaN,35.67,Yes,SKUOZ


In [56]:
import os

def is_same_noframes(exo_id, ego_id):
    rgb_path = f"../data/charades/CharadesEgo_v1_rgb"
    exo_rgb_dir = rgb_path + f"/{exo_id}"
    ego_rgb_dir = rgb_path + f"/{ego_id}"

    # Make sure both directories exist
    if not os.path.isdir(exo_rgb_dir) or not os.path.isdir(ego_rgb_dir):
        return False

    exo_frames = os.listdir(exo_rgb_dir)
    ego_frames = os.listdir(ego_rgb_dir)
    
    noframes_diff = len(exo_frames) - len(ego_frames)
    
    # Check if the number of frames in both directories are similar
    return abs(noframes_diff) == 0 # have the same number of frames
  
def get_ego_id_from_charades_video_id(df, video_id):
    ego_id = df[df['charades_video'] == video_id]['id']
    if len(ego_id) == 0:
      return None
    return ego_id.iloc[0] if isinstance(ego_id, pd.Series) else ego_id

In [ ]:
get_ego_id_from_charades_video_id(ego_df, "ZSHV8")

NameError: name 'ego_df' is not defined

In [57]:
ego_df['is_same_noframes'] = ego_df.apply(lambda row: is_same_noframes(row['id'][:-3], row['id']), axis=1)

In [58]:
ego_df[ego_df['is_same_noframes'] == True]['id'].nunique()

83

In [60]:
ego_df['id'].nunique()

3084

In [ ]:
ego_df[ego_df['is_same_noframes'] == True].head()

KeyboardInterrupt: 